# B9 · Spatial joins and overlays: landslides, districts and headquarters

> **Time:** about 2 hours  
> **You need:** Part A and Lessons B1 to B8, especially attribute joins with `.merge` (B6), spatial predicates (B7) and buffers in UTM 45N (B8).  
> **By the end you can:**
> 1. attach a district's attributes to every landslide point with a **spatial join** (`gpd.sjoin`), and choose between left, inner and right joins,
> 2. count points per polygon so that districts with no points show 0, and check a join against the original table,
> 3. cut polygons against each other with **overlays** (intersection, union, difference) and measure the new areas in km².

In Lesson B6 you joined two tables through a shared column, such as `province_code`. Often two layers share no column at all: a landslide point does not know which district it lies in. Its **location** does. This lesson joins and cuts layers by location, the heart of everyday GIS analysis. Remember that the landslide records are made-up practice data.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 19 of 21 · [Course home](../README.md)  
← Previous: [B8 · Buffers and nearest features: how close is close?](B08_Buffers_and_nearest_features.ipynb) · Next: [B10 · Counting points and density maps: where are the landslides?](B10_Counting_points_and_density_maps.ipynb) →

## 1. Two layers with no shared key

A **spatial join** joins two layers by where their features are, not by a shared column. A point gets the attributes of the polygon it falls in.

Before any join, prepare the data as in B6. The landslide file spells the district **Chitwan**, while the boundaries write **Chitawan**. As in B6, fix the spelling first, so that later checks compare like with like.

Both layers must also use the **same CRS**. Here both are in WGS84 (EPSG:4326), because you build the points from `lon` and `lat` columns.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("..") / "data"
districts = gpd.read_file(DATA / "nepal_districts.geojson")

slides = pd.read_csv(DATA / "practice_landslides.csv")
slides["district"] = slides["district"].replace("Chitwan", "Chitawan")   # as in B6

landslides = gpd.GeoDataFrame(
    slides,
    geometry=gpd.points_from_xy(slides["lon"], slides["lat"]),   # x = longitude, y = latitude
    crs="EPSG:4326",
)
print(len(districts), "districts and", len(landslides), "landslide records")
print(districts.crs == landslides.crs)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
districts.plot(ax=ax, color="whitesmoke", edgecolor="grey", linewidth=0.4)
landslides.plot(ax=ax, color="crimson", markersize=12)
ax.set_title("Practice landslide records (made-up data)")
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")
plt.show()

There are 77 districts and 40 landslide records, and the two CRS are equal (`True`). The points cluster in a few hill districts.

The landslide table already has a `district` column, typed by a person. The spatial join will find the district again from the coordinates. Two independent answers to the same question make a powerful check (step 5).

### Practice

**Q1 · one line:** Count how many records in `slides` now say `Chitawan`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
(slides["district"] == "Chitawan").sum()
```

9 records say `Chitawan`. Comparing a column with a value gives `True`/`False`, and `.sum()` counts the `True` values, as in A9.
</details>

**Q2 · two lines:** Read `district_headquarters.geojson` into `hq`. Then count how many headquarter names are also district names, with `.isin()` from B5.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
hq = gpd.read_file(DATA / "district_headquarters.geojson")
print(hq["name"].isin(districts["district"]).sum())
```

Only 22 of the 75 headquarter names match a district name. Bharatpur, for example, is the headquarters of Chitawan. Names cannot link these layers, but locations can.
</details>

**Q3 · three lines:** Make a map of the Bagmati districts (`province_code` `"NP03"`) with the landslides on top. As in A10, lines that only set up the plot (`fig, ax = plt.subplots()` and `plt.show()`) do not count toward the line count.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
bagmati = districts[districts["province_code"] == "NP03"]
fig, ax = plt.subplots()
bagmati.plot(ax=ax, color="whitesmoke", edgecolor="grey")
landslides.plot(ax=ax, color="crimson", markersize=12)
plt.show()
```

Points outside Bagmati are still drawn, so the map widens to include them. The axes follow all the layers you plot, not only the first one.
</details>

## 2. Your first spatial join with `sjoin`

`gpd.sjoin(left, right, how=..., predicate=...)` joins two GeoDataFrames by location.

- The **predicate** is the spatial test from B7. `predicate="within"` keeps a pair when the left feature lies within the right one.
- The result keeps the **left** layer's rows and geometry. Here every row is still a landslide point.
- The right layer's columns are added. `index_right` tells you which right-hand row matched.
- When both layers have a column with the same name, pandas adds the endings `_left` and `_right`. Both of ours have `district`.

**Predict:** each landslide lies in exactly one district. How many rows will the result have?

In [ ]:
joined = gpd.sjoin(landslides, districts, how="left", predicate="within")
print(joined.shape)
print(joined.columns.tolist())

In [ ]:
joined[["id", "district_left", "index_right", "district_right", "district_code", "province_code"]].head()

The result has 40 rows, one per landslide, and 13 columns. The eight landslide columns come first, then the district's columns.

`district_left` is the name typed in the CSV; `district_right` is the name the map gives. `index_right` is the row label of the matching district in `districts`, so the first row's `index_right` of 19 means `districts.loc[19]`, Dolakha.

### Practice

**Q1 · one line:** What geometry type does `joined` hold? Print the unique values of `joined.geom_type`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(joined.geom_type.unique())
```

Only `Point`. A spatial join never changes geometry: the left layer's points stay points, now carrying district attributes.
</details>

**Q2 · two lines:** Swap the two layers: join `districts` (left) to `landslides` (right) with `predicate="contains"`. Print the shape and the unique geometry types.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
swapped = gpd.sjoin(districts, landslides, how="inner", predicate="contains")
print(swapped.shape, swapped.geom_type.unique())
```

You get 40 rows again, but now they are district polygons. Chitawan appears 9 times, once per landslide inside it. The left layer always decides the geometry.
</details>

**Q3 · three lines:** Using `joined`, count the landslides in each `province_code`, then print the total of those counts.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
per_province = joined["province_code"].value_counts()
print(per_province)
print(per_province.sum())
```

NP03 (Bagmati) has 26, NP04 (Gandaki) 10 and NP01 (Koshi) 4, a total of 40. The province code came from the districts layer, not from the CSV.
</details>

## 3. Left, inner and right joins

The `how` argument decides which rows survive when a feature finds no partner. It works as in `.merge` (B6):

| `how` | Keeps | Geometry from |
| --- | --- | --- |
| `"left"` | every left feature; no match gives `NaN` | left |
| `"inner"` | only features that found a match | left |
| `"right"` | every right feature; no match gives `NaN` | right |

![Four Venn diagrams: left, right, inner and outer joins](../images/join_types.jpg)

*Diagram: the pyGIS project by Michael Mann (GPL-3.0). `sjoin` offers left, inner and right, but not outer.*

Below, the 75 headquarters are joined to the 13 Bagmati districts only. **Predict:** which join keeps all 75 points?

In [ ]:
hq = gpd.read_file(DATA / "district_headquarters.geojson")
bagmati = districts[districts["province_code"] == "NP03"]

left = gpd.sjoin(hq, bagmati, how="left", predicate="within")
inner = gpd.sjoin(hq, bagmati, how="inner", predicate="within")
print(len(hq), len(bagmati), len(left), len(inner))
inner[["name", "district"]].head()

In [ ]:
right = gpd.sjoin(hq, districts, how="right", predicate="within")
print(len(right), right.geom_type.unique())
right.loc[right["name"].isna(), ["district", "name"]]

The left join keeps all 75 headquarters; those outside Bagmati get `NaN` in the district columns. The inner join keeps only the 13 inside Bagmati.

The right join keeps all 77 districts as polygons. Two have no headquarters: **Nawalparasi East** and **Rukum West**. This points layer predates the split of Nawalparasi and Rukum (see the data README). A right join is a quick way to find such gaps.

### Practice

**Q1 · one line:** How many rows of `left` have no district? Use `.isna().sum()` on its `district` column.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
left["district"].isna().sum()
```

62 headquarters lie outside Bagmati, which is 75 − 13. In a left join, those rows stay, with `NaN` in every column from the right layer.
</details>

**Q2 · two lines:** Join `landslides` to `bagmati` with an inner join and `predicate="within"`. Print the number of rows and the unique values of `district_right`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
in_bagmati = gpd.sjoin(landslides, bagmati, how="inner", predicate="within")
print(len(in_bagmati), in_bagmati["district_right"].unique())
```

26 landslides fall in Bagmati, in four districts: Dolakha, Chitawan, Rasuwa and Sindhupalchok (the order may differ).
</details>

**Q3 · three lines:** Find the landslides **outside** Bagmati: make a left join of `landslides` to `bagmati`, keep the rows where `district_right` is missing, and print their `id` and `district_left`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
check = gpd.sjoin(landslides, bagmati, how="left", predicate="within")
outside = check[check["district_right"].isna()]
print(outside[["id", "district_left"]])
```

14 records are outside Bagmati, in Gorkha, Taplejung, Myagdi and Kaski. A left join followed by `.isna()` is the standard way to find features that did not match.
</details>

## 4. Counting points in each polygon

"How many landslides in each district?" is a spatial join followed by a `groupby` (A9).

1. Join points to polygons, so each point knows its district.
2. `groupby("district_code").size()` counts rows per district. `.reset_index(name="n_landslides")` turns that Series into a small table with a named count column.
3. Merge the counts back onto **all** districts with `how="left"`, as in B6. Districts without landslides get `NaN`.
4. `.fillna(0)` replaces `NaN` with 0, and `.astype(int)` makes whole numbers.

Step 4 matters. Without it, districts with no records vanish from the map, and "no records" looks like "no data".

In [ ]:
counts = joined.groupby("district_code").size().reset_index(name="n_landslides")
print(counts)

district_counts = districts.merge(counts, on="district_code", how="left")
district_counts["n_landslides"] = district_counts["n_landslides"].fillna(0).astype(int)
print(len(district_counts), district_counts["n_landslides"].sum())
print((district_counts["n_landslides"] == 0).sum(), "districts have no records")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
district_counts.plot(ax=ax, column="n_landslides", cmap="Reds", legend=True,
                     edgecolor="grey", linewidth=0.3,
                     legend_kwds={"label": "Landslide records per district", "shrink": 0.6})
ax.set_title("Practice landslide records per district (made-up data)")
ax.set_axis_off()
plt.show()

`counts` has 8 rows, one per district with records. After the merge, all 77 districts are back, and the counts still add up to 40. 69 districts show 0.

Always check that the total after counting equals the number of points you started with. If it is smaller, some points found no polygon; if it is larger, some points were counted twice.

`ax.set_axis_off()` hides the degree ticks, which add nothing to a choropleth. Lesson B11 covers this and other finishing touches.

### Practice

**Q1 · one line:** Show the 8 districts with records, from most to fewest: sort `district_counts` by `n_landslides` and show the `district` and `n_landslides` columns of the first 8 rows.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
district_counts.sort_values("n_landslides", ascending=False)[["district", "n_landslides"]].head(8)
```

Chitawan and Dolakha have 9 each, Myagdi and Rasuwa 5, Gorkha and Taplejung 4, Sindhupalchok 3 and Kaski 1.
</details>

**Q2 · two lines:** Instead of counting, add up the landslide area per district: group `joined` by `district_right`, sum `area_m2`, and print the result sorted from largest to smallest.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
area_m2 = joined.groupby("district_right")["area_m2"].sum()
print(area_m2.sort_values(ascending=False))
```

Chitawan has the largest total, 79,970 m², and Kaski the smallest, 250 m². The same join supports any summary: count, sum, mean or max.
</details>

**Q3 · three lines:** Count only the **earthquake** landslides per district: filter `joined` by `trigger`, then group by `district_right` and count with `.size()`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
quakes = joined[joined["trigger"] == "earthquake"]
quake_counts = quakes.groupby("district_right").size()
print(quake_counts)
```

Chitawan, Dolakha and Gorkha have one earthquake record each (practice data). Filter first, then count: the join itself does not change.
</details>

## 5. Checking a join against the table

A spatial join ignores spelling: only the coordinates decide. The landslide table also has a district typed by a person. Comparing the two is an excellent check.

If both agree, the coordinates and the names support each other. If they disagree, either a point is misplaced or a name is spelled differently.

**Predict:** after the Chitwan fix in step 1, how many of the 40 records will agree? And how many would disagree without the fix?

In [ ]:
same = joined["district_left"] == joined["district_right"]
print(same.sum(), "of", len(joined), "records agree")

In [ ]:
raw = pd.read_csv(DATA / "practice_landslides.csv")     # the file as it is, no fix
raw_points = gpd.GeoDataFrame(raw, geometry=gpd.points_from_xy(raw["lon"], raw["lat"]), crs="EPSG:4326")
raw_joined = gpd.sjoin(raw_points, districts, how="left", predicate="within")

mismatch = raw_joined["district_left"] != raw_joined["district_right"]
print(mismatch.sum(), "records disagree")
raw_joined.loc[mismatch, ["id", "district_left", "district_right"]].head(3)

After the fix, all 40 records agree. Without it, 9 records disagree, and every one says `Chitwan` against `Chitawan`.

Notice that the spatial join itself worked in both cases: the points landed in Chitawan either way. Only the text comparison failed. That is why you fix spellings before any check or attribute join.

### Practice

**Q1 · one line:** Did any landslide fall outside every district? Count the missing values in `joined["district_right"]`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
joined["district_right"].isna().sum()
```

0: every point lies inside a district. A point in the wrong hemisphere, for example with `lat` and `lon` swapped, would show up here as missing.
</details>

**Q2 · two lines:** Print the unique CSV spellings in `raw_joined` that disagree with the map. Use the `mismatch` mask from the example.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
bad_names = raw_joined.loc[mismatch, "district_left"].unique()
print(bad_names)
```

Only `Chitwan` disagrees. A short list of unique values is easier to act on than 9 rows.
</details>

**Q3 · three lines:** Check the provinces too. Read `province_names.csv`, merge it onto `joined` by `province_code`, and count the rows where the CSV's province name equals the map's. Both tables have a `province` column, so `.merge` names them `province_x` (from `joined`) and `province_y`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
names = pd.read_csv(DATA / "province_names.csv")
checked = joined.merge(names, on="province_code")
print((checked["province_x"] == checked["province_y"]).sum())
```

All 40 agree. `.merge` adds `_x` and `_y` to clashing names, as `sjoin` adds `_left` and `_right`.
</details>

## 6. Overlay: cutting polygons with intersection

A spatial join copies attributes and leaves the shapes alone. An **overlay** cuts the shapes themselves and makes new polygons.

`gpd.overlay(df1, df2, how="intersection")` keeps only the ground covered by **both** layers. Each new piece carries the attributes of both parents.

![Intersection: only the overlap C of square A and circle B is kept](../images/vector_intersection.jpg)

*Diagram: the pyGIS project by Michael Mann (GPL-3.0).*

You will measure areas, so first project both layers to UTM 45N (EPSG:32645), as in B8. Below, each headquarters first gets its district's name through a spatial join. Then its 10 km buffer is cut by the district borders. `.rename(columns={old: new})` gives a column a clearer name.

In [ ]:
districts_utm = districts.to_crs(32645)

hq_in = gpd.sjoin(hq, districts, how="inner", predicate="within")      # each HQ learns its district
buffers = hq_in[["name", "district", "geometry"]].to_crs(32645)
buffers = buffers.rename(columns={"district": "hq_district"})
buffers["geometry"] = buffers.buffer(10000)                              # 10 km in metres
print(len(buffers), buffers.crs.to_epsg())

In [ ]:
pieces = gpd.overlay(buffers, districts_utm, how="intersection")
pieces["area_km2"] = pieces.area / 1000000
print(len(pieces), "pieces")
pieces.loc[pieces["name"] == "Bharatpur", ["name", "hq_district", "district", "area_km2"]]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
bharatpur = pieces[pieces["name"] == "Bharatpur"]
bharatpur.plot(ax=ax, column="district", cmap="Set2", legend=True, edgecolor="black")
districts_utm.plot(ax=ax, facecolor="none", edgecolor="grey", linewidth=0.6)
ax.set_xlim(bharatpur.total_bounds[0] - 20000, bharatpur.total_bounds[2] + 20000)
ax.set_ylim(bharatpur.total_bounds[1] - 20000, bharatpur.total_bounds[3] + 20000)
ax.set_title("Bharatpur's 10 km buffer cut by district borders")
ax.set_axis_off()
plt.show()

The 75 buffers became 138 pieces. Bharatpur's buffer was cut into 3: about 250.3 km² in Chitawan, 49.0 km² in Nawalparasi East and 14.4 km² in Tanahu.

Each piece has the columns of both layers: `name` and `hq_district` from the buffer, `district` and the codes from the district. To keep only the part of each buffer inside its **own** district, compare the two district columns:

In [ ]:
own = pieces[pieces["hq_district"] == pieces["district"]]
print(len(own), round(own["area_km2"].max(), 1))
own.sort_values("area_km2")[["name", "district", "area_km2"]].head(3)

`own` has 75 rows, one per headquarters. A full 10 km buffer covers about 313.7 km² here (a little less than π × 10² ≈ 314.2 km², because the circle is drawn with straight segments, B8). Jumlikhalanga in Rukum East keeps only about 75.6 km² of its buffer inside its own district, and Lalitpur about 80.6 km².

### Practice

**Q1 · one line:** Into how many pieces was Kathmandu's buffer cut? Count the rows of `pieces` whose `name` is `"Kathmandu"`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
(pieces["name"] == "Kathmandu").sum()
```

5 pieces: Kathmandu's 10 km circle reaches into Bhaktapur, Dhading, Lalitpur and Nuwakot as well as Kathmandu.
</details>

**Q2 · two lines:** Keep the pieces that lie **outside** their own headquarters' district (`hq_district` differs from `district`), then print how many there are.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
spill = pieces[pieces["hq_district"] != pieces["district"]]
print(len(spill))
```

63 pieces spill over a district border, and 75 + 63 = 138. Many headquarters lie less than 10 km from a neighbouring district.
</details>

**Q3 · three lines:** What share of Bhaktapur district lies within 10 km of its headquarters? Take the area of Bhaktapur's row in `own`, divide by the area of Bhaktapur in `districts_utm` (in km²), and print the percentage rounded to 1 decimal.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
near_km2 = own.loc[own["district"] == "Bhaktapur", "area_km2"].iloc[0]
district_km2 = districts_utm.loc[districts_utm["district"] == "Bhaktapur"].area.iloc[0] / 1000000
print(round(near_km2 / district_km2 * 100, 1))
```

100.0%: the whole of Bhaktapur district (about 122.6 km² in this data) is within 10 km of its headquarters. It is the smallest district, smaller than one buffer.
</details>

## 7. Union and difference

Two more `how` options cut the layers in other ways:

- `how="union"` keeps **all** the ground of both layers, cut into pieces. Pieces covered by only one layer get `NaN` for the other layer's columns.
- `how="difference"` keeps the ground of the first layer that the second layer does **not** cover. Only the first layer's columns remain. GIS programs call this **erase**.

![Union: all of square A and circle B, cut into pieces](../images/vector_union.jpg) ![Difference: square A with circle B cut away](../images/vector_erase.jpg)

*Diagrams: the pyGIS project by Michael Mann (GPL-3.0).*

**Predict:** the union of Chitawan and Bharatpur's buffer will have three pieces. What will they be?

In [ ]:
chitawan = districts_utm.loc[districts_utm["district"] == "Chitawan", ["district", "geometry"]]
bharatpur_buffer = buffers.loc[buffers["name"] == "Bharatpur", ["name", "geometry"]]

united = gpd.overlay(chitawan, bharatpur_buffer, how="union")
united["area_km2"] = united.area / 1000000
united.drop(columns="geometry")

In [ ]:
far = gpd.overlay(districts_utm, buffers, how="difference")
far["area_km2"] = far.area / 1000000
print(len(far), far.columns.tolist())
print(round(far["area_km2"].sum()), "km² lie more than 10 km from every headquarters")

fig, ax = plt.subplots(figsize=(9, 5))
far.plot(ax=ax, color="tan", edgecolor="white", linewidth=0.3)
ax.set_title("Nepal with every 10 km headquarters buffer erased")
ax.set_axis_off()
plt.show()

The union has three rows: the overlap (both names filled in, about 250.3 km²), the rest of Chitawan (`name` is `NaN`, about 1,990.3 km²) and the part of the buffer outside Chitawan (`district` is `NaN`, about 63.4 km²).

The difference keeps all 77 districts with only their own columns, each with holes where buffers were. About 126,296 km² of these simplified boundaries lie more than 10 km from every headquarters.

### Practice

**Q1 · one line:** Using `far`, print the district with the largest area far from any headquarters. Sort by `area_km2` and look at the last row.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
far.sort_values("area_km2")[["district", "area_km2"]].tail(1)
```

Dolpa, with about 7,652 km² more than 10 km from its headquarters. Large mountain districts have one headquarters and a lot of remote ground.
</details>

**Q2 · two lines:** Run the union of `bharatpur_buffer` (first) and `chitawan` (second). Print the number of rows and the columns. What changed compared with the example?

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
swapped_union = gpd.overlay(bharatpur_buffer, chitawan, how="union")
print(len(swapped_union), swapped_union.columns.tolist())
```

Still 3 pieces with the same shapes, but now `name` comes before `district`. Union is symmetrical; only the column order follows the order of the layers.
</details>

**Q3 · three lines:** Run a difference the other way round: Bharatpur's buffer minus Chitawan. Print the number of rows and its area in km², rounded to 1 decimal.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
outside_chitawan = gpd.overlay(bharatpur_buffer, chitawan, how="difference")
print(len(outside_chitawan))
print(round(outside_chitawan.area.sum() / 1000000, 1))
```

One row of about 63.4 km²: the part of the buffer in Nawalparasi East and Tanahu (49.0 + 14.4 km²). Unlike union, difference is not symmetrical: the order of the layers decides what survives.
</details>

## 8. `keep_geom_type` and checking areas after an overlay

Two neighbouring districts share a border but no ground. Their intersection is therefore a **line**, not a polygon.

`keep_geom_type=True` keeps only results of the same type as the first layer (here polygons) and drops such lines. `keep_geom_type=False` keeps everything. If you leave it out, GeoPandas behaves like `True` and prints a warning when it drops something.

After an overlay, check the areas. The pieces of a district should add up to the whole district. Overlapping inputs break this rule: where two buffers overlap, the same ground is counted twice. Merging the buffers first with `.union_all()` (B8) avoids that.

In [ ]:
kathmandu = districts_utm[districts_utm["district"] == "Kathmandu"]
lalitpur = districts_utm[districts_utm["district"] == "Lalitpur"]

only_polygons = gpd.overlay(kathmandu, lalitpur, how="intersection", keep_geom_type=True)
everything = gpd.overlay(kathmandu, lalitpur, how="intersection", keep_geom_type=False)
print(len(only_polygons), len(everything))
print(everything.geom_type.iloc[0], round(everything.length.iloc[0] / 1000, 1), "km of shared border")

In [ ]:
merged = gpd.GeoDataFrame(geometry=[buffers.union_all()], crs=buffers.crs)   # one shape, no overlaps
near = gpd.overlay(districts_utm, merged, how="intersection")

near_km2 = near.area.sum() / 1000000
far_km2 = far.area.sum() / 1000000
total_km2 = districts_utm.area.sum() / 1000000
print(round(near_km2), round(far_km2), round(near_km2 + far_km2), round(total_km2))
print(round(pieces["area_km2"].sum()), "km² if you add up the overlapping pieces")

Kathmandu and Lalitpur share about 23.1 km of border. With `keep_geom_type=True` that line is dropped, so the result is empty (0 rows).

With merged buffers, near and far add up to the whole country: about 21,721 + 126,296 = 148,017 km², the total you met in B4. Adding up `pieces` instead gives about 22,595 km², because ground near two headquarters (in the Kathmandu Valley, for example) is counted twice.

### Practice

**Q1 · one line:** Run the intersection of `lalitpur` and `kathmandu` (in that order) with `keep_geom_type=False`, and print its geometry types.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(gpd.overlay(lalitpur, kathmandu, how="intersection", keep_geom_type=False).geom_type.tolist())
```

A single `MultiLineString`: their shared border, made of several line parts. Swapping the layers does not change the shape.
</details>

**Q2 · two lines:** Check the areas for Chitawan alone: add the area of Chitawan's rows in `pieces` and in `far` (both in km²), and compare with Chitawan's area in `districts_utm`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
parts_km2 = pieces.loc[pieces["district"] == "Chitawan", "area_km2"].sum() + far.loc[far["district"] == "Chitawan", "area_km2"].sum()
print(round(parts_km2, 1), round(districts_utm.loc[districts_utm["district"] == "Chitawan"].area.iloc[0] / 1000000, 1))
```

Both are about 2,240.5 km². Only Bharatpur's buffer reaches into Chitawan, so nothing overlaps and the check works without merging.
</details>

**Q3 · three lines:** Do the same check for Kathmandu district. Add its rows in `pieces` and in `far`, and print the total next to the district's real area. Why is the total too large?

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
k_pieces_km2 = pieces.loc[pieces["district"] == "Kathmandu", "area_km2"].sum()
k_far_km2 = far.loc[far["district"] == "Kathmandu", "area_km2"].sum()
print(round(k_pieces_km2 + k_far_km2, 1), round(kathmandu.area.iloc[0] / 1000000, 1))
```

About 641.8 km² against a real 414.3 km². The Kathmandu, Lalitpur and Bhaktapur buffers overlap inside Kathmandu district, so their pieces count the same ground two or three times.
</details>

## 9. When it breaks: a column renamed by the join

Both `landslides` and `districts` have a column called `district`. After the join you ask for `district`. Run the cell.

In [ ]:
joined["district"]

Scroll to the last line: `KeyError: 'district'`. As in A9, a `KeyError` means there is no column with that name.

The column exists in both input layers, but the join renamed it to `district_left` and `district_right` to keep both. This surprises everyone the first time.

The fix: run `print(joined.columns.tolist())` after every join, then use the exact new name. If you prefer other endings, `sjoin` accepts `lsuffix="csv", rsuffix="map"`.

**Try it:** change the cell to show `district_right`.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · Landslides per province, zeros included

Read `nepal_provinces.geojson` and `province_names.csv`. Join `landslides` to the provinces with `sjoin`, count records per `province_code`, merge the counts back onto all 7 provinces and fill missing counts with 0. Add the province names and print the name and count of every province.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
provinces = gpd.read_file(DATA / "nepal_provinces.geojson")
names = pd.read_csv(DATA / "province_names.csv")

in_province = gpd.sjoin(landslides, provinces, how="inner", predicate="within")
prov_counts = in_province.groupby("province_code").size().reset_index(name="n_landslides")

provinces = provinces.merge(prov_counts, on="province_code", how="left")
provinces["n_landslides"] = provinces["n_landslides"].fillna(0).astype(int)
provinces = provinces.merge(names, on="province_code")
print(provinces[["province", "n_landslides"]])
```

Koshi has 4, Bagmati 26 and Gandaki 10; the other four provinces show 0 instead of disappearing. The counts add up to 40. Attribute joins (B6) and spatial joins work together in one short workflow.
</details>

### Review 2 · Which headquarters serve which province?

Join `hq` to `districts` with an inner join. Count the headquarters per `province_code`. Then use a right join to list every district without a headquarters, with its province code.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
hq_joined = gpd.sjoin(hq, districts, how="inner", predicate="within")
print(hq_joined.groupby("province_code").size())

all_districts = gpd.sjoin(hq, districts, how="right", predicate="within")
no_hq = all_districts[all_districts["name"].isna()]
print(no_hq[["district", "province_code"]])
```

The counts add up to 75. Nawalparasi East (NP04) and Rukum West (NP06) have no headquarters in this older layer. Before using any layer, ask when it was made.
</details>

### Review 3 · How much of each Bagmati district is near its headquarters?

Using `own` from step 6, keep the Bagmati rows (`province_code` `"NP03"`). Merge on the full district area in km² from `districts_utm`, compute the percentage of each district within 10 km of its own headquarters, and print the districts sorted from highest to lowest share.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
bagmati_own = own[own["province_code"] == "NP03"][["district", "area_km2"]]

full = districts_utm[["district"]].copy()
full["district_km2"] = districts_utm.area / 1000000

share = bagmati_own.merge(full, on="district")
share["pct_near"] = (share["area_km2"] / share["district_km2"] * 100).round(1)
print(share.sort_values("pct_near", ascending=False)[["district", "pct_near"]])
```

Bhaktapur leads with 100.0% and Kathmandu follows with 54.5%. Chitawan, Sindhuli and Makawanpur have the smallest shares, 11.2% to 12.5%. A buffer covers about the same area everywhere, so the share mostly reflects district size.
</details>

### Review 4 · Landslides near a headquarters, by spatial join

In B8 you found landslides within 10 km of a headquarters with buffers. Do it now with a spatial join: project `landslides` to UTM 45N, join them to `buffers` with `predicate="within"` (inner), and count the **unique** landslide `id`s. Then print how many pairs the join returned, and explain the difference.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
landslides_utm = landslides.to_crs(32645)
near_hq = gpd.sjoin(landslides_utm, buffers, how="inner", predicate="within")

print(len(near_hq), "pairs")
print(near_hq["id"].nunique(), "unique landslides")
print(near_hq["name"].value_counts())
```

`.nunique()` counts distinct values. Both numbers are 9 (5 near Dhunche, 3 near Bharatpur, 1 near Charikot), so no landslide lies in two overlapping buffers. Had one done so, it would appear once per buffer: always count unique IDs after a join that can match more than once.
</details>

## Recap

- `gpd.sjoin(left, right, how=..., predicate=...)` joins by location. The left layer keeps its geometry; the right layer's columns are added, with `_left`/`_right` endings on clashing names.
- `how="left"` keeps every left feature, `"inner"` only matches, and `"right"` every right feature with the right layer's geometry.
- Count points per polygon with `sjoin` → `groupby(...).size()` → `merge(how="left")` → `fillna(0)`, and check that the total still equals the number of points.
- Fix spellings first (Chitwan → Chitawan), then compare the joined names with the typed ones: a join you have not checked is a guess.
- `gpd.overlay` cuts shapes: `intersection` keeps shared ground, `union` keeps all, `difference` erases. Measure areas in UTM 45N, and merge overlapping inputs before adding areas.

### Words you learned

| Word | Meaning |
| --- | --- |
| spatial join | Joining two layers by where their features are, not by a shared column |
| `gpd.sjoin` | GeoPandas' spatial join function |
| predicate | The spatial test a join uses, such as `within` or `intersects` |
| `index_right` | Column added by `sjoin` that holds the row label of the matching right-hand feature |
| suffix | An ending such as `_left` or `_right` added to column names that clash |
| left / inner / right join | Which rows survive a join: all left rows, only matches, or all right rows |
| `reset_index(name=...)` | Turns a counted Series into a table with a named count column |
| `fillna` | Replaces missing values (`NaN`) with a value you choose |
| overlay | An operation that cuts polygons against each other and makes new shapes |
| intersection | The ground covered by both layers |
| union (overlay) | All the ground of both layers, cut into pieces |
| difference / erase | The ground of the first layer that the second does not cover |
| `keep_geom_type` | Whether an overlay keeps only results of the first layer's geometry type |
| `.rename(columns=...)` | Gives columns new names |

## Next

**B10 · Counting points and density maps:** count landslides in square and hexagon grid cells, turn counts into densities per 100 km², and see how the choice of map classes changes the story a map tells.

<!-- pygile-nav -->
**PyGILE Course** · lesson 19 of 21 · [Course home](../README.md)  
← Previous: [B8 · Buffers and nearest features: how close is close?](B08_Buffers_and_nearest_features.ipynb) · Next: [B10 · Counting points and density maps: where are the landslides?](B10_Counting_points_and_density_maps.ipynb) →